# Kazakh Profanity Detection: Colab demo

Runs in a **fresh Google Colab runtime**.

* **Part A** downloads the trained model and predicts on your own text, with no training.
* **Part B** (optional) reproduces data preparation, training and test evaluation. Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*. Training takes a few minutes.

Before running, set `REPO_URL` and `HF_MODEL_ID` in the first cell.

In [ ]:
REPO_URL = "https://github.com/zhanel00/kazakh-profanity-detection.git"
HF_MODEL_ID = "<your-hf-user>/kazakh-profanity-xlmr"   # trained model on the Hugging Face Hub

!git clone -q $REPO_URL repo
%cd repo
# Colab already ships torch; install the remaining pinned packages
!pip -q install transformers==5.6.2 tokenizers==0.22.2 sentencepiece==0.2.1 huggingface_hub==1.12.0 accelerate==1.13.0 scikit-learn==1.6.1

## Part A: predict with the saved model

In [ ]:
from src.predict import Predictor
clf = Predictor(HF_MODEL_ID)   # or Predictor('artifacts/model') after Part B

examples = [
    'Өте керемет ойын, маған қатты ұнады!',
    'Жарнама тым көп, ойын нашар, қатып қала береді',
    'Қотақбас ойын екен',
    'Мал ойын тартылмайды',
    'zaebal reklama',
]
for r in clf(examples):
    print(r['label'], f"P(profane)={r['prob_profane']:.3f}", '|', r['text'])

In [ ]:
#@title Enter your own comment
text = "Сапасы жақсы, бағасы арзан"  #@param {type:"string"}
r = clf(text)
print('label =', r['label'], '(1 = profane)', '| P(profane) =', r['prob_profane'], '| threshold =', r['threshold'])

## Part B (optional): reproduce the full pipeline

`data/processed/` is committed, so `prepare_data` is only needed if you have the raw file in `data/raw/`.

In [ ]:
# !python -m src.prepare_data          # needs data/raw/kazakh_profanity_share.csv
!python -m src.train                    # fine-tunes XLM-R, saves artifacts/model
!python -m src.evaluate                 # one evaluation on the held-out test set

In [ ]:
from IPython.display import Image, display
import pandas as pd, json
print(json.dumps(json.load(open('results/test_metrics.json')), indent=1)[:800])
display(Image('results/confusion_matrix_test.png'))
pd.read_csv('results/test_errors.csv').head(20)